In [7]:
import subprocess
import time
import itertools
import os
import sys
import pandas as pd
import glob
from datetime import datetime

# ================= 設定區 =================
dic = {
    'batch_size': [1000],
    'epoch': [1000],
    'layer': [4, 3, 2],
    'hidden': [32, 16],
    'data_version': [58],            
    'lr': [0.003, 0.001, 0.03, 0.01],
    'column': ['acceleration_X,acceleration_Y,acceleration_Z,gyroscope_X,gyroscope_Y,gyroscope_Z'],
    'folds': [1, 2, 3, 4, 5],
    # ==== 邊緣端校正與測試路徑 ====
    'target_version': ['58/special_data_hengling_yi'],  # 使用者前5分鐘校正數據
    'test_version': ['58/test_data_hengling_yi'],     # 實際測試推論數據
    'lambda_dann': [1.0]                    # 保留相容槽
}

MAX_CONCURRENT_JOBS = 4  
TRAIN_SCRIPT = "train.py" 
TEST_SCRIPT = "test.py"   
OUTPUT_LOG_DIR = "./test_log"
# =========================================

def get_combinations(params):
    keys = list(params.keys())
    values = list(params.values())
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

def run_phase(phase_name, script_name, combinations, max_jobs):
    print(f"\n=== 開始執行階段: {phase_name} ===")
    total_jobs = len(combinations)
    running_processes = []
    
    for i, p in enumerate(combinations):
        cmd = [
            'python3', script_name,
            f'--batch_size={p["batch_size"]}',
            f'--epoch={p["epoch"]}',
            f'--layer={p["layer"]}',
            f'--hidden={p["hidden"]}',
            f'--data_version={p["data_version"]}',
            f'--lr={p["lr"]}',
            f'--column={p["column"]}',
            f'--fold={p["folds"]}', 
            f'--lambda_dann={p["lambda_dann"]}',
            f'--target_version={p["target_version"]}'
        ]
        
        # 若為測試階段，需額外傳入 test_version
        if phase_name == "Testing":
            cmd.append(f'--test_version={p["test_version"]}')
        
        proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL)
        running_processes.append(proc)
        
        if i % 10 == 0:
            print(f"[{phase_name}] 進度: {i}/{total_jobs} (Running: {len(running_processes)})")

        while len(running_processes) >= max_jobs:
            running_processes = [proc for proc in running_processes if proc.poll() is None]
            if len(running_processes) >= max_jobs:
                time.sleep(1)

    for proc in running_processes:
        proc.wait()
    print(f"=== {phase_name} 階段完成 ===\n")

def collect_results():
    print(f"=== 正在從 {OUTPUT_LOG_DIR} 彙整最新測試報告 ===")
    
    # [更新] 修改搜尋關鍵字以適配最新 Deep CORAL 版本
    result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*_DeepCoral_Adapted_result.csv"))
    
    # 若找不到，向下相容尋找所有 CSV
    if not result_files:
        result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*.csv"))
        if not result_files or any("Final_Report" in f for f in result_files):
            print(f"在 {OUTPUT_LOG_DIR} 找不到任何有效的測試結果檔案。")
            return

    all_dfs = []
    for f in result_files:
        if "Final_Report" in f: continue 
        try:
            df = pd.read_csv(f)
            if not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"讀取 {f} 失敗: {e}")
    
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        
        # [更新] 使用最終完全體的指標 (adapt_macro_f1) 作為排序依據
        sort_target = "adapt_macro_f1"
        if sort_target in final_df.columns:
            final_df = final_df.sort_values(by=sort_target, ascending=False)
        elif "macro_f1" in final_df.columns:
            final_df = final_df.sort_values(by="macro_f1", ascending=False) # 備用相容
            
        out_name = f"Final_Report_DeepCoral_v{dic['data_version'][0]}.csv"
        final_df.to_csv(out_name, index=False)
        
        print("-" * 50)
        print(f"報告整合完成！共匯總 {len(all_dfs)} 筆測試數據。")
        print(f"最終報告已產出: {out_name}")
        print("-" * 50)
        print("Top 5 最佳模型表現 (依據 Deep CORAL + 動態閾值 F1 排序):")
        
        # [更新] 挑選出四個狀態中最具代表性的欄位來印出 (Base vs Adapt)
        try:
            display_cols = [
                'run_name', 
                'base_macro_f1', 'adapt_macro_f1', 
                'base_acc', 'adapt_acc',
                'adapt_f1_notTired', 'adapt_f1_Tired', 'adapt_f1_Other'
            ]
            print(final_df.head(5)[display_cols].to_string(index=False))
        except KeyError as e:
            print(f"[提示] 部分指定欄位不存在，直接印出前五筆全資料: {e}")
            print(final_df.head(5))
            
    else:
        print("沒有有效的數據可以合併。")

def main():
    start_time = datetime.now()
    combinations = list(get_combinations(dic)) 
    
    run_phase("Training", TRAIN_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    run_phase("Testing", TEST_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    collect_results()

    end_time = datetime.now()
    print(f"總耗時: {end_time - start_time}")

if __name__ == "__main__":
    main()


=== 開始執行階段: Training ===
[Training] 進度: 0/120 (Running: 1)
[Training] 進度: 10/120 (Running: 4)
[Training] 進度: 20/120 (Running: 4)
[Training] 進度: 30/120 (Running: 3)
[Training] 進度: 40/120 (Running: 3)
[Training] 進度: 50/120 (Running: 4)
[Training] 進度: 60/120 (Running: 4)
[Training] 進度: 70/120 (Running: 4)
[Training] 進度: 80/120 (Running: 4)
[Training] 進度: 90/120 (Running: 4)
[Training] 進度: 100/120 (Running: 4)
[Training] 進度: 110/120 (Running: 4)
=== Training 階段完成 ===


=== 開始執行階段: Testing ===
[Testing] 進度: 0/120 (Running: 1)


Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x32 and 6x6)
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~

[Testing] 進度: 10/120 (Running: 3)


Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x32 and 6x6)
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x32 and 6x6)
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_i

[Testing] 進度: 20/120 (Running: 1)


...

[Testing] 進度: 100/120 (Running: 1)


Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x16 and 6x6)
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_a

[Testing] 進度: 110/120 (Running: 3)


Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_inv @ coral_cov_src_sqrt
              ~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x16 and 6x6)
RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x16 and 6x6)
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 421, in <module>
    main()
    ~~~~^^
  File "/home/mluser/114_NCUT_A100/v38/test.py", line 255, in main
    W_align = cov_tgt_sqrt_i

=== Testing 階段完成 ===

=== 正在從 ./test_log 彙整最新測試報告 ===
--------------------------------------------------
報告整合完成！共匯總 120 筆測試數據。
最終報告已產出: Final_Report_DeepCoral_v58.csv
--------------------------------------------------
Top 5 最佳模型表現 (依據 Deep CORAL + 動態閾值 F1 排序):
                                                 run_name  base_macro_f1  adapt_macro_f1  base_acc  adapt_acc  adapt_f1_notTired  adapt_f1_Tired  adapt_f1_Other
fold1_layer_2_hidden_16_lr_0.003_ServerMaster_EdgeAdapted       0.618160        0.645831  0.811404   0.872807           0.864600        0.119403        0.953488
 fold4_layer_2_hidden_16_lr_0.01_ServerMaster_EdgeAdapted       0.593154        0.636764  0.747076   0.850877           0.831386        0.104167        0.974740
 fold3_layer_4_hidden_32_lr_0.03_ServerMaster_EdgeAdapted       0.539816        0.632189  0.673977   0.811404           0.770870        0.153846        0.971852
 fold5_layer_3_hidden_32_lr_0.03_ServerMaster_EdgeAdapted       0.607655        0.631273  0.771